# Module 1 · LLM Basics & Tokens
What is an LLM? How does tokenization work? Your first API call.

---
> **Setup:** Run the first two cells before anything else.
> API key is loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # picks up modules/.env symlink
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'gemma-4-31b-it'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                # parse suggested retry delay from error (e.g. 'retryDelay': '30s')
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen    = client.models.generate_content
_raw_stream = client.models.generate_content_stream
_raw_embed  = client.models.embed_content
_raw_count  = client.models.count_tokens
client.models.generate_content        = lambda *a,**kw: _call_with_retry(_raw_gen,    *a,**kw)
client.models.generate_content_stream = lambda *a,**kw: _call_with_retry(_raw_stream, *a,**kw)
client.models.embed_content           = lambda *a,**kw: _call_with_retry(_raw_embed,  *a,**kw)
client.models.count_tokens            = lambda *a,**kw: _call_with_retry(_raw_count,  *a,**kw)

print('\u2705 Setup complete. Model:', MODEL, '| Retry-on-429: enabled')


✅ Setup complete. Model: gemma-4-31b-it | Retry-on-429: enabled


### 🔌 API Key Test

In [3]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

✅ API key working!
Model says: Hello LLM course


---
## 1. What Is an LLM?

An LLM (Large Language Model) is, at heart, a **next-token predictor** trained on massive amounts of text.

At every step it asks: *"Given everything so far, what token most likely comes next?"*

```
Input text
    │
    ▼
┌─────────────┐
│  Tokenizer  │  "The cat sat" → [450, 4097, 2492]
└──────┬──────┘
       │
       ▼
┌─────────────┐
│ Transformer │  Billions of parameters, trained on the internet
│   (the LLM) │
└──────┬──────┘
       │
       ▼
┌────────────────────────────────┐
│ Probability distribution over │  " on" → 42%
│ the entire vocabulary (~32k    │  " in" → 18%
│ possible next tokens)          │  " near" → 9%  ...
└──────────────┬─────────────────┘
               │
               ▼
         Sample a token → append → repeat until done
```

**Developer analogy:** Think of it as autocomplete on steroids — trained on the whole internet. It doesn't "know" facts the way a database does; it learned statistical patterns so well that it *acts as if* it knows them.

| Concept | What it means in practice |
|---|---|
| Probabilistic output | Same prompt → different answer each time (unless `temperature=0`) |
| Context window | Model can only "see" a fixed number of tokens at once |
| No memory by default | Each API call is stateless; you send the whole conversation each time |
| Thinking models | Some models (like Gemma 4) reason internally before answering — improves quality |

---
## 2. Tokens & Tokenization

Tokens are **not** words, and not characters — they're chunks roughly 3–4 characters long on average.

**Why it matters:** you pay per token; context windows are measured in tokens; speed depends on token count.

In [4]:
sentences = [
    "Hello",
    "Hello, world!",
    "Hello, world! How are you doing today?",
    "Supercalifragilisticexpialidocious",
    "The quick brown fox jumps over the lazy dog.",
    "Python is great for data science, machine learning, and automation.",
    "🚀🔥💡",
    "\n\n\n\n",
]

print(f"{'Text':<60} {'Tokens':>6}")
print("-" * 68)
for s in sentences:
    result = client.models.count_tokens(model=MODEL, contents=s)
    display_s = repr(s) if len(s) < 55 else repr(s[:52]) + "..."
    print(f"{display_s:<60} {result.total_tokens:>6}")

Text                                                         Tokens
--------------------------------------------------------------------
'Hello'                                                           2


'Hello, world!'                                                   5
'Hello, world! How are you doing today?'                         11


'Supercalifragilisticexpialidocious'                             11
'The quick brown fox jumps over the lazy dog.'                   11


'Python is great for data science, machine learning, '...        14
'🚀🔥💡'                                                             4


'\n\n\n\n'                                                        2


In [5]:
# ✏️  Change this string and re-run to experiment
my_text = "Write your own sentence here and see how many tokens it uses!"

result = client.models.count_tokens(model=MODEL, contents=my_text)
tokens = result.total_tokens
print(f"Text   : {my_text}")
print(f"Words  : {len(my_text.split())}")
print(f"Chars  : {len(my_text)}")
print(f"Tokens : {tokens}")
print(f"Ratio  : {len(my_text)/tokens:.1f} chars/token  |  {len(my_text.split())/tokens:.1f} words/token")

Text   : Write your own sentence here and see how many tokens it uses!


Words  : 12
Chars  : 61
Tokens : 14
Ratio  : 4.4 chars/token  |  0.9 words/token


---
## 3. Your First API Call

In [6]:
response = client.models.generate_content(
    model=MODEL,
    contents="What is a neural network? Explain in 2 sentences for a software developer.",
    config=cfg()
)
print(get_text(response))

A neural network is a series of layered, interconnected nodes that act as a non-linear function approximator to map complex inputs to specific outputs. It learns the optimal internal weights by iteratively minimizing a loss function using backpropagation and gradient descent.


In [7]:
# Dissect the response object
print("=" * 60)
print("RESPONSE OBJECT ANATOMY")
print("=" * 60)

text = get_text(response)
print("\n--- text ---")
print(text[:200], "..." if len(text) > 200 else "")

print("\n--- finish_reason ---")
print(f"  {response.candidates[0].finish_reason}")

print("\n--- usage_metadata ---")
u = response.usage_metadata
print(f"  prompt_token_count    : {u.prompt_token_count}")
print(f"  candidates_token_count: {u.candidates_token_count}")
print(f"  thoughts_token_count  : {getattr(u, 'thoughts_token_count', 'N/A')}")
print(f"  total_token_count     : {u.total_token_count}")

RESPONSE OBJECT ANATOMY

--- text ---
A neural network is a series of layered, interconnected nodes that act as a non-linear function approximator to map complex inputs to specific outputs. It learns the optimal internal weights by iterat ...

--- finish_reason ---
  FinishReason.STOP

--- usage_metadata ---
  prompt_token_count    : 17
  candidates_token_count: 48
  thoughts_token_count  : 352
  total_token_count     : 417
